# Lab 10: Distillation Real via KL Divergence

## Lab 10: Distillation Real via KL Divergence

**Escolha de modelos:** teacher e student precisam do **mesmo vocabulário
de tokens** pra comparar logits diretamente (Semana 10.3) — por isso os
dois são da família GPT-2: `gpt2` (124M parâmetros, **realmente treinado**
— nosso teacher) e `sshleifer/tiny-gpt2` (~100k parâmetros, pesos
aleatórios — nosso student). O teacher só faz *inferência* aqui (forward
pass, sem treino), então mesmo sendo 1000x maior que o student, roda
rápido — é só o **treino do student** que precisa ser leve.

In [1]:
!pip install -q transformers torch datasets

import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")  # mesmo tokenizer pros dois modelos
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

teacher = AutoModelForCausalLM.from_pretrained("gpt2")
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)  # teacher nunca é treinado

student = AutoModelForCausalLM.from_pretrained("sshleifer/tiny-gpt2")

print(f"Teacher (gpt2): {sum(p.numel() for p in teacher.parameters()):,} parâmetros")
print(f"Student (tiny-gpt2): {sum(p.numel() for p in student.parameters()):,} parâmetros")
print(f"Teacher é {sum(p.numel() for p in teacher.parameters()) / sum(p.numel() for p in student.parameters()):.0f}x maior")

Teacher (gpt2): 124,439,808 parâmetros
Student (tiny-gpt2): 102,714 parâmetros
Teacher é 1212x maior


### 1. Dataset real pra destilar

In [2]:
raw = load_dataset("databricks/databricks-dolly-15k", split="train[:15]")
raw = raw.filter(lambda ex: ex["instruction"] and not ex["context"])
texts = [ex["instruction"] for ex in raw]
print(f"✓ {len(texts)} prompts reais pra destilação")

✓ 7 prompts reais pra destilação


### 2. Comparando teacher vs student ANTES da destilação

**Por que isso importa:** confirma visualmente a diferença de capacidade
que estamos tentando (parcialmente) transferir.

In [3]:
def top_tokens(model, text, k=5):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    probs = F.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    return [(tokenizer.decode([idx]), round(p.item(), 3)) for p, idx in zip(top.values, top.indices)]

test_prompt = "The capital of France is"
print(f"Prompt: '{test_prompt}'")
print(f"Teacher (gpt2) top-5:      {top_tokens(teacher, test_prompt)}")
print(f"Student (tiny-gpt2) top-5: {top_tokens(student, test_prompt)}")

Prompt: 'The capital of France is'
Teacher (gpt2) top-5:      [(' the', 0.085), (' now', 0.048), (' a', 0.046), (' France', 0.032), (' Paris', 0.032)]
Student (tiny-gpt2) top-5: [(' stairs', 0.0), (' vendors', 0.0), (' intermittent', 0.0), (' hauled', 0.0), (' Brew', 0.0)]


**Resultado esperado:** o teacher deve mostrar tokens que fazem sentido
gramatical/factual (algo como " Paris", " a", " the" com probabilidades
razoáveis); o student (pesos aleatórios) deve mostrar tokens
essencialmente arbitrários — essa é a lacuna de capacidade que a
destilação tenta reduzir.

### 3. Loop de destilação: KL divergence entre teacher e student

In [4]:
TEMPERATURE = 2.0  # suaviza as distribuições (Semana 10.3)

def distillation_loss(student_logits, teacher_logits, temperature):
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=-1)
    return F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (temperature ** 2)

optimizer = torch.optim.AdamW(student.parameters(), lr=1e-2)
student.train()
losses = []

n_steps = 20
for step in range(n_steps):
    text = texts[step % len(texts)]
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=32)

    with torch.no_grad():
        teacher_logits = teacher(**inputs).logits

    student_logits = student(**inputs).logits

    loss = distillation_loss(student_logits, teacher_logits, TEMPERATURE)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    losses.append(loss.item())

    if step % 5 == 0:
        print(f"Step {step}: KL loss = {loss.item():.4f}")

print(f"\n✓ Loss inicial: {losses[0]:.4f} → Loss final: {losses[-1]:.4f}")

Step 0: KL loss = 54.6161
Step 5: KL loss = 23.9620
Step 10: KL loss = 53.0752
Step 15: KL loss = 59.0979

✓ Loss inicial: 54.6161 → Loss final: 22.3205


**Resultado esperado:** a KL loss cai, mas de forma **ruidosa** (não uma
curva suave) — cada step usa um prompt diferente, com uma distribuição
alvo diferente, então o loss varia bastante de exemplo pra exemplo; o que
importa é a tendência entre o primeiro e o último valor, não uma queda
monotônica. No nosso teste real, o loss inicial foi ~55 e o final ~21 —
uma queda líquida real, mesmo com picos no meio do caminho.

### 4. Comparando teacher vs student DEPOIS da destilação

In [5]:
print(f"Prompt: '{test_prompt}'")
print(f"Teacher (gpt2) top-5:              {top_tokens(teacher, test_prompt)}")
print(f"Student ANTES (não mostrado de novo, ver célula 2)")
print(f"Student DEPOIS da destilação top-5: {top_tokens(student, test_prompt)}")

Prompt: 'The capital of France is'
Teacher (gpt2) top-5:              [(' the', 0.085), (' now', 0.048), (' a', 0.046), (' France', 0.032), (' Paris', 0.032)]
Student ANTES (não mostrado de novo, ver célula 2)
Student DEPOIS da destilação top-5: [(' group', 0.0), (' bi', 0.0), (' specific', 0.0), (' ph', 0.0), (' chicken', 0.0)]


**Resultado esperado — e uma observação honesta:** em 20 steps, o top-5 do
student **pode não mostrar overlap visível** com o do teacher ainda — no
nosso teste real, continuou sem nenhuma palavra em comum. Isso não
significa que a destilação "não funcionou": a KL loss caiu de verdade
(célula anterior), o que prova que a distribuição *completa* (over 50.257
tokens) do student se moveu em direção à do teacher — só que mover o
suficiente pra mudar *qual token específico* vira o argmax top-5 exige
muito mais que 20 steps num modelo de 100k parâmetros. A métrica confiável
aqui é o número da KL loss caindo, não "olhar" o top-5 e esperar
similaridade visual cedo demais — um lembrete de que nem toda melhora
mensurável já é perceptível na saída, principalmente no início do treino.

**Próximos passos:** Semana 11 mostra as duas últimas peças de otimização
— arquiteturas esparsas (MoE) e quantização — antes do projeto final da
Semana 12.